In [ ]:
# imports 
import numpy as np
from typing import Callable

In [ ]:
# SA v2
def sim_annealing_v2(hx_func:Callable, theta_names:list[str], theta_domains:list[list[int | float]]):

    # reflection logic for handling thetas generated outside of domain
    def reflect(val, low, high):
        while val > high or val < low:
            if val > high:
                val = high - (val - high)
            if val < low:
                val = low + (low - val)
        return val

    # initalizations
    temp = 1
    dif = 1  # difference (for stopping rule)
    iter = 0  # starting iterations value
    min_iter = 100
    max_iter = 5000
    acc_history = []  # for tracking acceptance history
    target_low = 0.2
    target_high = 0.6  # upper bound for acceptance rate
    domain_widths = np.array([high - low for low, high in theta_domains])  # domain width is unique for each theta
    factor = domain_widths.copy()
    sc_vec = np.minimum(domain_widths * 0.10, np.clip(factor, 0.01 * domain_widths, domain_widths) * np.sqrt(temp))  # scaler vector is factor * sqrt(temp), but cannot be larger than 1/10 of domain width and is unique per theta
    num_thetas = len(theta_names)
    max_proposals = 200  # max num of proposed theta values before we move on to next iteration
    proposals = 0  

    # starting values
    str_vals = {}  # start location
    for theta, value in zip(theta_names, theta_domains):
        start = np.random.uniform(low=value[0], high=value[1], size=1)  # generates random value within theta domain
        start = np.round(start, decimals=2) 
        start = start[0]  # turns 1-value array into a single float
        str_vals[theta] = start  # adds theta start value to str_vals dict
    str_ht = hx_func(**str_vals)  # start height

    # theta and height history
    theta_hist = np.zeros((max_iter + 1, num_thetas))  # creates a 2D array with pre-allocated size, has size max_iter+1 and has a collumn for each theta
    theta_hist[0] = [str_vals[name] for name in theta_names]  # assigns first theta values as our starting point
    ht_hist = [str_ht]
    best_theta = theta_hist[0].copy()  # tracks theta based on best global height
    best_height = str_ht  # tracks gobal best height

    # main loop
    while dif > 0.00001 and iter < max_iter:
        proposals = 0

        # acceptance logic: accept when move is higher and if lower, accept it with probaility exp(dht / temp)
        while True:
            zeta = sc_vec * np.random.normal(loc=0, scale=1, size=num_thetas)  # returns list of zetas for each theta, generated from N(0, 1)
            prop_theta = theta_hist[iter] + zeta  # new proposed theta, adjusted by zeta value
            prop_theta = np.array([
                reflect(val, low, high)
                for val, (low, high) in zip(prop_theta, theta_domains)
            ])  # reflection function applied if proposed theta is outside of domain

            prop_theta_dict = dict(zip(theta_names, prop_theta))  # combines theta_names and prop_theta lists into a single dict for function call
            hprop = hx_func(**prop_theta_dict)  # calculate height of proposed thetas
            dht = hprop - ht_hist[iter]  # delta height = proposed - current
            cur_odds = np.random.rand()  # generates currend odds value, aka our current luck

            proposals += 1
            if cur_odds <= np.exp(dht / temp) or proposals >= max_proposals:  # metropolis criterion acceptance logic: p_accept = exp(dht/temp)
                if proposals >= max_proposals:
                    prop_theta = theta_hist[iter]  # if proposal limit reached without acceptable thetas, then keep old theta values
                    hprop = ht_hist[iter]
                break

        # once acceptable theta and height are found, we record it
        theta_hist[iter + 1] = prop_theta  # records and appends new theta value to history
        ht_hist.append(hprop)
        iter += 1

        # if proposed height > best known height, then record it
        if hprop > best_height:
            best_height = hprop
            best_theta = prop_theta.copy()

        # check if theta actually moved and record it
        moved = bool(np.any(theta_hist[iter] != theta_hist[iter - 1]))
        acc_history.append(moved)
        
        # temperature adjustment
        if iter > 100:
            recent = acc_history[-100:]
            acc_rate = np.mean(recent)  # acceptance rate is the avg of the times theta moved vs not. since T is 1 and F is 0, finding mean is to find the % of T

            # increase size of jumps if acceptance rate is low
            factor = factor / 1.5 if acc_rate < target_low else factor
            # decrease size of jumps if acceptance rance is high
            factor = factor * 1.5 if acc_rate > target_high else factor
            factor = np.clip(factor, 0.01 * domain_widths, domain_widths)

        # stopping rule
        if iter > min_iter:
            current_max = np.max(ht_hist)
            first_half_max = np.max(ht_hist[0:int(np.floor(iter/2))])  # the difference is the max height minus max height in first half of iterations
            denominator = max(abs(first_half_max), 1.0)   # avoid div-by-zero
            dif = (current_max - first_half_max) / denominator  # its first half because if we do 2nd half, it might give us the local max in case that the temp is too low and the SA is stuck

        # temperature and scaler update
        temp = 1 / np.log(1 + iter * 0.5)  # 0.5 can be adjusted to speed up or slow down cooling, or can be changed to 1 / (1 + iter)^2 for quadratic cooling
        sc_vec = np.minimum(domain_widths * 0.10, np.clip(factor, 0.01 * domain_widths, domain_widths) * np.sqrt(temp))  # scaler is factor * sqrt(temp), but cannot be larger than 1/10 of domain width

    # output values
    last_theta = theta_hist[iter] # last value for theta
    last_height = ht_hist[-1]  # last height value

    return last_theta, last_height, best_theta, best_height

# Ex formula:
def cross_wave(x, y):
    val = -(x * np.sin(20 * y) + y * np.sin(20 * x))**2 * np.cosh(np.sin(10 * x) * x) \
          -(x * np.cos(10 * y) - y * np.sin(10 * x))**2 * np.cosh(np.cos(20 * y) * y)
    return val
# Cross-wave global max = 0

# Ex usage:
list_1 = ['x', 'y']
domain = [[-1, 1], [-1, 1]]
sim_annealing_v2(hx_func=cross_wave, theta_names=list_1, theta_domains=domain)
